# Multiple Linear Regression — Area, Bedrooms, Age vs Price

---

**Project:** ML with Scikit-Learn  
**Notebook:** 02 — Multiple Linear Regression  
**Algorithm:** Multiple Linear Regression (3 features, 1 label)  
**Author:** Ather-ops  

---

## Objective

Extend the simple linear regression pipeline from Notebook 01 by adding two more features — **Bedrooms** and **Age** — alongside **Area** to predict house **Price**. More features means a richer model that captures more of what drives price variation.

---

## From Simple to Multiple

| Notebook | Features Used | Model Equation |
|----------|--------------|----------------|
| 01 — Simple | Area only | $\hat{Price} = w_1 \cdot Area + b$ |
| 02 — Multiple | Area, Bedrooms, Age | $\hat{Price} = w_1 \cdot Area + w_2 \cdot Bedrooms + w_3 \cdot Age + b$ |

Each weight $w_i$ tells us: holding all other features constant, how much does Price change for a one-unit increase in feature $i$?

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Create dataset |
| 3 | Exploratory Data Analysis — 3 scatter plots |
| 4 | Descriptive statistics and missing value check |
| 5 | Outlier detection and treatment |
| 6 | Correlation analysis |
| 7 | Feature and target selection |
| 8 | Train-test split |
| 9 | Feature scaling |
| 10 | Model training |
| 11 | Predictions |
| 12 | Model evaluation |
| 13 | Feature importance — learned coefficients |
| 14 | Post-training visualizations |
| 15 | Actual vs Predicted plot |
| 16 | Residual analysis |
| 17 | Predict price of a new house |
| 18 | New house prediction visualization |

---
## Step 1 — Import Libraries

Same libraries as Notebook 01 — no new imports needed. Multiple linear regression uses exactly the same Scikit-Learn class (`LinearRegression`) as simple linear regression. The only difference is the shape of `X`.

| Library | Role |
|---------|------|
| `pandas` | DataFrame creation, correlation matrix |
| `numpy` | Numerical operations, new house input |
| `matplotlib.pyplot` | All plots and visualizations |
| `train_test_split` | 80/20 data split |
| `LinearRegression` | Multiple linear regression model |
| `StandardScaler` | Normalize all 3 features |
| `mean_squared_error` | Average squared prediction error |
| `mean_absolute_error` | Average absolute prediction error |
| `r2_score` | Variance in Price explained by the model |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## Step 2 — Create Dataset

We create a dataset with 25 houses and 4 columns — three input features and one target.

| Column | Type | Range | Description |
|--------|------|-------|-------------|
| `Area` | int | 600 – 4500 | House size in square feet |
| `Bedrooms` | int | 1 – 6 | Number of bedrooms |
| `Age` | int | 1 – 20 | Age of the house in years |
| `Price` | int | 100 – 700 | Sale price in thousands — target |

**Expected relationships:**
- Larger `Area` → higher `Price`
- More `Bedrooms` → higher `Price` (generally)
- Older `Age` → lower `Price`

The dataset includes some noise — not every relationship is perfectly linear — which is realistic.

In [ ]:
# Create housing dataset — 25 houses, 3 features, 1 label
data = {
    "Area":     [600,  800,  950,  1100, 1200, 1350, 1500, 1650, 1800, 2000,
                 2200, 2400, 2550, 2700, 2900, 3100, 3300, 3500, 3700, 3900,
                 4000, 4100, 4300, 4400, 4500],
    "Bedrooms": [1,    2,    2,    2,    3,    3,    3,    3,    4,    4,
                 4,    4,    5,    5,    5,    5,    5,    6,    6,    6,
                 6,    6,    6,    6,    6],
    "Age":      [15,   12,   10,   18,   8,    14,   6,    20,   5,    11,
                 3,    9,    7,    16,   4,    13,   2,    10,   6,    8,
                 1,    5,    3,    7,    2],
    "Price":    [110,  145,  162,  175,  200,  218,  240,  230,  275,  310,
                 345,  370,  395,  380,  430,  455,  490,  520,  555,  580,
                 610,  625,  655,  670,  700]
}

df = pd.DataFrame(data)

print("="*65)
print("Dataset — 25 Houses, 3 Features, 1 Target")
print("="*65)
print(df.to_string(index=True))
print("="*65)
print("Shape:", df.shape)

---
## Step 3 — Exploratory Data Analysis — 3 Scatter Plots

We plot each of the three features against `Price` separately. This is the standard EDA step for multiple regression — it shows which features have a strong visual relationship with the target and which may add less signal.

**What to look for in each plot:**

| Plot | Expected pattern | Strong signal if |
|------|-----------------|------------------|
| Area vs Price | Clear upward trend | Points cluster tightly around a rising line |
| Bedrooms vs Price | Stepwise upward pattern | Higher bedroom counts consistently = higher prices |
| Age vs Price | Downward trend | Older houses cluster at lower prices |

In [ ]:
# EDA — scatter plots for all 3 features vs Price
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Area vs Price
axes[0].scatter(df["Area"], df["Price"],
                color="steelblue", edgecolors="white", s=70, linewidth=0.8)
axes[0].set_xlabel("Area (sq ft)", fontsize=11)
axes[0].set_ylabel("Price (thousands)", fontsize=11)
axes[0].set_title("Area vs Price (Before Training)", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Plot 2: Bedrooms vs Price
axes[1].scatter(df["Bedrooms"], df["Price"],
                color="darkorange", edgecolors="white", s=70, linewidth=0.8)
axes[1].set_xlabel("Bedrooms", fontsize=11)
axes[1].set_ylabel("Price (thousands)", fontsize=11)
axes[1].set_title("Bedrooms vs Price (Before Training)", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.5)

# Plot 3: Age vs Price
axes[2].scatter(df["Age"], df["Price"],
                color="seagreen", edgecolors="white", s=70, linewidth=0.8)
axes[2].set_xlabel("Age (years)", fontsize=11)
axes[2].set_ylabel("Price (thousands)", fontsize=11)
axes[2].set_title("Age vs Price (Before Training)", fontsize=11)
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Exploratory Data Analysis — Feature vs Target",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("="*65)

---
## Step 4 — Descriptive Statistics and Missing Value Check

Before any cleaning or modeling, we inspect the statistical properties of each column.

**Key checks for this dataset:**

| Column | Healthy min | Healthy max | Flag if |
|--------|------------|------------|--------|
| Area | > 0 | < 10000 | Negative or unrealistically large |
| Bedrooms | 1 | 10 | Zero or above 10 |
| Age | 0 | 100 | Negative |
| Price | > 0 | Depends | Negative or extreme compared to 75th percentile |

A gap between `mean` and `50%` (median) in the Price column signals skew — the IQR method handles this in Step 5.

In [ ]:
# Descriptive statistics and missing value check
print("Basic Statistics:")
print("="*65)
print(df.describe().round(2))
print("="*65)
print("Missing Values per Column:")
print(df.isnull().sum())
print("="*65)
print("Data Types:")
print(df.dtypes)
print("="*65)
print("Dataset shape:", df.shape)

---
## Step 5 — Outlier Detection and Treatment

We apply the **IQR method** to the `Price` column — the target variable — to detect and clip extreme values.

**Why focus on Price?**  
Outliers in the target variable distort the regression line far more than outliers in features. A single house priced at 5x the normal range pulls the entire line toward it, inflating errors for every other house.

**Formula:**

$$IQR = Q3 - Q1$$
$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

**Treatment — clipping:**  
`np.clip()` replaces any value outside the fences with the fence boundary. No rows are deleted — sample size is preserved.

In [ ]:
# IQR outlier detection on Price
Q1  = df["Price"].quantile(0.25)
Q3  = df["Price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Outlier Detection — Price Column")
print("="*65)
print(f"Q1            : {Q1}")
print(f"Q3            : {Q3}")
print(f"IQR           : {IQR}")
print(f"Lower Bound   : {lower_bound:.2f}")
print(f"Upper Bound   : {upper_bound:.2f}")
print("="*65)

outliers = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)]
print(f"Outliers detected : {len(outliers)}")
if len(outliers) > 0:
    print(outliers)
print("="*65)

# Clip
df["Price"] = np.clip(df["Price"], lower_bound, upper_bound)
print("Clipping applied. Price column cleaned.")
print("="*65)

---
## Step 5b — Outlier Visualization

We visualize the Price distribution using a boxplot and histogram to confirm the clipping worked correctly.

- The boxplot shows Q1, median, Q3, and whiskers at the IQR fences
- The histogram shows frequency distribution of prices after cleaning
- Dashed lines mark the computed lower and upper bounds

In [ ]:
# Outlier visualization — boxplot and histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(df["Price"], patch_artist=True,
                boxprops=dict(facecolor="steelblue", color="navy"),
                medianprops=dict(color="red", linewidth=2),
                whiskerprops=dict(color="navy"),
                capprops=dict(color="navy"))
axes[0].axhline(lower_bound, color="orange", linestyle="--",
                linewidth=1.2, label=f"Lower: {lower_bound:.0f}")
axes[0].axhline(upper_bound, color="green",  linestyle="--",
                linewidth=1.2, label=f"Upper: {upper_bound:.0f}")
axes[0].set_ylabel("Price (thousands)")
axes[0].set_title("Price Boxplot — IQR Fences")
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Histogram
axes[1].hist(df["Price"], bins=8, color="steelblue", edgecolor="white", linewidth=0.8)
axes[1].axvline(lower_bound, color="orange", linestyle="--",
                linewidth=1.5, label="Lower Bound")
axes[1].axvline(upper_bound, color="green",  linestyle="--",
                linewidth=1.5, label="Upper Bound")
axes[1].set_xlabel("Price (thousands)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Price Distribution After Clipping")
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*65)

---
## Step 6 — Correlation Analysis

Before modeling, we compute the **Pearson correlation** between every pair of columns. This is a number between -1 and +1:

| Value | Meaning |
|-------|---------|
| Close to +1 | Strong positive relationship |
| Close to -1 | Strong negative relationship |
| Close to 0 | Weak or no linear relationship |

**What we look for:**
- High correlation between a feature and `Price` → the feature is predictive → keep it
- High correlation between two features → **multicollinearity** → the model may have unstable weights

The heatmap visualizes the full correlation matrix — darker color = stronger relationship.

In [ ]:
# Correlation matrix
corr = df.corr(numeric_only=True).round(3)

print("Correlation Matrix:")
print("="*65)
print(corr)
print("="*65)

# Heatmap using matplotlib
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

cols = corr.columns.tolist()
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, fontsize=10)
ax.set_yticklabels(cols, fontsize=10)

# Annotate cells
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if abs(corr.values[i, j]) > 0.5 else "black")

ax.set_title("Feature Correlation Heatmap", fontsize=12)
plt.tight_layout()
plt.show()
print("="*65)

---
## Step 7 — Feature and Target Selection

We now use all three features as inputs:

| Variable | Columns | Shape | Role |
|----------|---------|-------|------|
| `X` | Area, Bedrooms, Age | (25, 3) | Input feature matrix — 3 columns |
| `Y` | Price | (25,) | Target vector |

**Difference from Notebook 01:**  
In simple linear regression `X` had shape `(20, 1)`. Here `X` has shape `(25, 3)`. The `LinearRegression` class handles both cases identically — it learns one weight per column automatically.

In [ ]:
# Feature matrix and target vector
X = df[["Area", "Bedrooms", "Age"]]
Y = df["Price"]

print("Feature and Target Selection")
print("="*65)
print("Features (X) shape  :", X.shape)
print("Target (Y) shape    :", Y.shape)
print("="*65)
print("X — first 5 rows:")
print(X.head().to_string())
print("="*65)
print("Y — first 5 values:")
print(Y.head().to_string())
print("="*65)

---
## Step 8 — Train-Test Split

We split 25 houses into training and test sets:

| Parameter | Value | Houses |
|-----------|-------|--------|
| `test_size=0.2` | 20% | 5 houses for testing |
| `random_state=42` | Fixed seed | Same split every run |
| Training set | 80% | 20 houses for training |

The model trains on 20 houses across all 3 features and is evaluated on 5 completely unseen houses.

In [ ]:
# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("Train-Test Split")
print("="*65)
print(f"Total houses   : {len(X)}")
print(f"Training set   : {len(X_train)} houses")
print(f"Test set       : {len(X_test)} houses")
print("="*65)
print("X_train shape  :", X_train.shape)
print("X_test shape   :", X_test.shape)
print("Y_train shape  :", Y_train.shape)
print("Y_test shape   :", Y_test.shape)
print("="*65)

---
## Step 9 — Feature Scaling

With three features on different scales, scaling is now critical — not just good practice:

| Feature | Typical range | Without scaling |
|---------|--------------|----------------|
| `Area` | 600 – 4500 | Dominates — large raw values |
| `Bedrooms` | 1 – 6 | Underweighted — small values |
| `Age` | 1 – 20 | Moderate — may be drowned out by Area |

`StandardScaler` transforms each feature independently to mean = 0, std = 1:

$$z = \frac{x - \mu}{\sigma}$$

After scaling, all three features contribute to the model on equal numerical footing. The weights the model learns reflect actual predictive importance — not just raw magnitude.

In [ ]:
# Feature scaling — fit on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("StandardScaler Applied")
print("="*65)
print("Learned means per feature  :", scaler.mean_.round(2))
print("Learned scales per feature :", scaler.scale_.round(2))
print("="*65)
print("X_train_scaled — first 5 rows:")
scaled_df = pd.DataFrame(X_train_scaled,
                          columns=["Area_s", "Bedrooms_s", "Age_s"])
print(scaled_df.head().round(4).to_string())
print("="*65)
print("Post-scaling mean (should be ~0):", X_train_scaled.mean(axis=0).round(6))
print("Post-scaling std  (should be ~1):", X_train_scaled.std(axis=0).round(6))
print("="*65)

---
## Step 10 — Model Training

We train `LinearRegression` on the 3-feature scaled training matrix. The model learns four values:

| Parameter | Notation | Meaning |
|-----------|----------|--------|
| Weight for Area | $w_1$ | Price change per 1-unit increase in scaled Area |
| Weight for Bedrooms | $w_2$ | Price change per 1-unit increase in scaled Bedrooms |
| Weight for Age | $w_3$ | Price change per 1-unit increase in scaled Age |
| Intercept | $b$ | Base price when all scaled features equal zero |

A **negative** $w_3$ for Age is expected — older houses are worth less. A **positive** $w_1$ for Area is expected — bigger houses are worth more.

In [ ]:
# Model training
model = LinearRegression()
model.fit(X_train_scaled, Y_train)

features = ["Area", "Bedrooms", "Age"]

print("Model Trained — Learned Parameters")
print("="*65)
print(f"Intercept (b) : {model.intercept_:.4f}")
print("="*65)
for feat, coef in zip(features, model.coef_):
    direction = "(positive — increases price)" if coef > 0 else "(negative — decreases price)"
    print(f"  w_{feat:10s}: {coef:+.4f}  {direction}")
print("="*65)
print("Equation:")
print(f"  Price = {model.coef_[0]:.4f}*Area_scaled",
      f"+ {model.coef_[1]:.4f}*Bedrooms_scaled",
      f"+ {model.coef_[2]:.4f}*Age_scaled",
      f"+ {model.intercept_:.4f}")

---
## Step 11 — Predictions

We generate predicted prices for all 5 test houses and print a comparison table showing the actual price, predicted price, and the prediction error for each house.

The error column (Actual - Predicted) gives a row-by-row sense of model performance before looking at aggregate metrics. Positive error = model underestimated. Negative error = model overestimated.

In [ ]:
# Predictions on test set
y_pred = model.predict(X_test_scaled)

# Comparison table
results = pd.DataFrame({
    "Area"           : X_test["Area"].values,
    "Bedrooms"       : X_test["Bedrooms"].values,
    "Age"            : X_test["Age"].values,
    "Actual Price"   : Y_test.values,
    "Predicted Price": y_pred.round(2),
    "Error"          : (Y_test.values - y_pred).round(2)
})

print("Prediction Results — Test Set")
print("="*80)
print(results.to_string(index=False))
print("="*80)

---
## Step 12 — Model Evaluation

We compute four standard regression metrics on the test set:

| Metric | Formula | Unit | Best value |
|--------|---------|------|------------|
| MSE | $\frac{1}{n}\sum(y-\hat{y})^2$ | Price$^2$ | As low as possible |
| RMSE | $\sqrt{MSE}$ | Price (thousands) | As low as possible |
| MAE | $\frac{1}{n}\sum|y-\hat{y}|$ | Price (thousands) | As low as possible |
| R2 | $1 - \frac{SS_{res}}{SS_{tot}}$ | 0 to 1 | As close to 1 as possible |

**Comparing with Notebook 01:**  
The R2 score here should be higher than in the simple linear regression notebook. Adding Bedrooms and Age gives the model more information to work with — it can explain price variation that Area alone cannot capture.

In [ ]:
# Model evaluation
MSE      = mean_squared_error(Y_test, y_pred)
RMSE     = np.sqrt(MSE)
MAE      = mean_absolute_error(Y_test, y_pred)
R2       = r2_score(Y_test, y_pred)

print("Model Evaluation — Test Set (Multiple Linear Regression)")
print("="*65)
print(f"MSE      : {MSE:.4f}")
print(f"RMSE     : {RMSE:.4f}  (price units — thousands)")
print(f"MAE      : {MAE:.4f}  (price units — thousands)")
print(f"R2 Score : {R2:.4f}")
print("="*65)
print(f"The model explains {R2*100:.1f}% of price variation")
print(f"using Area, Bedrooms, and Age together.")
print("="*65)

---
## Step 13 — Feature Importance — Learned Coefficients

Because we scaled all features before training, the magnitude of each weight directly reflects how much that feature contributes to predicting price — on a comparable scale.

**How to read this:**
- The feature with the **largest absolute coefficient** contributes most to price predictions
- A **positive coefficient** means: as this feature increases, predicted price increases
- A **negative coefficient** means: as this feature increases, predicted price decreases

We visualize the coefficients as a horizontal bar chart. This is the simplest form of feature importance available for linear models.

In [ ]:
# Feature importance — learned coefficients
features  = ["Area", "Bedrooms", "Age"]
coefs     = model.coef_
colors    = ["steelblue" if c > 0 else "tomato" for c in coefs]

print("Learned Coefficients (on scaled features):")
print("="*65)
for feat, coef in zip(features, coefs):
    print(f"  {feat:12s}: {coef:+.4f}")
print("="*65)

plt.figure(figsize=(7, 4))
bars = plt.barh(features, coefs, color=colors, edgecolor="white", height=0.5)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")

for bar, coef in zip(bars, coefs):
    plt.text(
        coef + (2 if coef > 0 else -2),
        bar.get_y() + bar.get_height() / 2,
        f"{coef:+.2f}",
        va="center", ha="left" if coef > 0 else "right",
        fontsize=10
    )

plt.xlabel("Coefficient Value (scaled features)", fontsize=11)
plt.title("Feature Importance — Learned Weights", fontsize=12)
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*65)

---
## Step 14 — Post-Training Visualization

We plot each feature against actual and predicted prices on the test set — the same scatter view as EDA but now with the model's predictions overlaid.

**Reading the plots:**

| Element | Color | Meaning |
|---------|-------|--------|
| Actual prices | Blue | Real test set prices |
| Predicted prices | Red | What the model predicted for the same houses |
| Tight overlap | — | Model predictions are close to reality |
| Wide separation | — | Model is struggling with that feature's range |

Note that these plots show one feature at a time on the x-axis, but the model used all three simultaneously to generate each prediction. The x-axis is just for positioning the dots.

In [ ]:
# Post-training visualization — actual vs predicted per feature
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

feature_cols   = ["Area", "Bedrooms", "Age"]
x_labels       = ["Area (sq ft)", "Bedrooms", "Age (years)"]
actual_colors  = ["steelblue", "steelblue", "steelblue"]
pred_colors    = ["red", "red", "red"]

for i, (feat, xlabel) in enumerate(zip(feature_cols, x_labels)):
    axes[i].scatter(X_test[feat], Y_test,
                    color=actual_colors[i], edgecolors="white",
                    s=90, linewidth=0.8, label="Actual", zorder=3)
    axes[i].scatter(X_test[feat], y_pred,
                    color=pred_colors[i], edgecolors="white",
                    s=90, linewidth=0.8, marker="D", label="Predicted", zorder=4,
                    alpha=0.8)
    axes[i].set_xlabel(xlabel, fontsize=11)
    axes[i].set_ylabel("Price (thousands)", fontsize=11)
    axes[i].set_title(f"{feat} vs Price (After Training)", fontsize=11)
    axes[i].legend(fontsize=9)
    axes[i].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Actual vs Predicted — All 3 Features", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("="*65)

---
## Step 15 — Actual vs Predicted Plot

The most compact evaluation plot for any regression model. Actual prices on the X-axis, predicted prices on the Y-axis:

- A **perfect model** places all dots on the dashed diagonal line where Actual = Predicted
- **Points above** the diagonal: model underestimated the price
- **Points below** the diagonal: model overestimated the price
- **R2 score** is directly proportional to how tightly the dots cluster around the diagonal

In [ ]:
# Actual vs Predicted scatter
plt.figure(figsize=(6, 5))
plt.scatter(Y_test, y_pred,
            color="steelblue", edgecolors="white",
            s=120, linewidth=0.8, zorder=3, label="Test houses")

# Perfect prediction diagonal
min_val = min(float(Y_test.min()), float(y_pred.min())) - 15
max_val = max(float(Y_test.max()), float(y_pred.max())) + 15
plt.plot([min_val, max_val], [min_val, max_val],
         color="red", linestyle="--", linewidth=1.5,
         label="Perfect prediction", zorder=2)

plt.xlabel("Actual Price (thousands)", fontsize=12)
plt.ylabel("Predicted Price (thousands)", fontsize=12)
plt.title(f"Actual vs Predicted  |  R2 = {R2:.4f}", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*65)

---
## Step 16 — Residual Analysis

Residuals = Actual Price minus Predicted Price. We plot residuals against each feature and as a histogram.

**What a good residual plot looks like:**
- Dots scattered randomly above and below the zero line — no pattern
- No funnel shape (which would indicate non-constant variance)
- No curve (which would indicate a non-linear relationship the model missed)

The histogram should approximate a bell curve centered at zero — normally distributed residuals are one of the core assumptions of linear regression.

In [ ]:
# Residual analysis
residuals = Y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals,
                color="steelblue", edgecolors="white",
                s=100, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("Predicted Price (thousands)", fontsize=11)
axes[0].set_ylabel("Residual (Actual - Predicted)", fontsize=11)
axes[0].set_title("Residuals vs Predicted Values", fontsize=12)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Residual histogram
axes[1].hist(residuals, bins=6, color="steelblue",
             edgecolor="white", linewidth=0.8)
axes[1].axvline(0, color="red", linestyle="--",
                linewidth=1.5, label="Zero error")
axes[1].set_xlabel("Residual", fontsize=11)
axes[1].set_ylabel("Frequency", fontsize=11)
axes[1].set_title("Residual Distribution", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*65)
print(f"Mean residual    : {residuals.mean():.4f}  (should be close to 0)")
print(f"Std of residuals : {residuals.std():.4f}")
print("="*65)

---
## Step 17 — Predict Price of a New House

We now use the trained model to predict the price of a brand new house — one that was never in the dataset.

**New house profile:**

| Feature | Value | Context |
|---------|-------|--------|
| Area | 2600 sq ft | Medium-large house |
| Bedrooms | 4 | Family home |
| Age | 5 years | Relatively new |

**Prediction steps:**
1. Create input array with shape `(1, 3)` — one row, three features in the same order as training
2. Scale using `scaler.transform()` — applies training-set statistics, never re-fits
3. Pass through `model.predict()` to get the price

> The order of features must match exactly what was used during training: Area first, then Bedrooms, then Age.

In [ ]:
# New house prediction
new_house        = np.array([[2600, 4, 5]])   # Area=2600, Bedrooms=4, Age=5
new_house_scaled = scaler.transform(new_house)
predicted_price  = model.predict(new_house_scaled)

print("New House Prediction")
print("="*65)
print(f"Area           : {new_house[0][0]} sq ft")
print(f"Bedrooms       : {new_house[0][1]}")
print(f"Age            : {new_house[0][2]} years")
print("="*65)
print(f"Predicted Price: {predicted_price[0]:.2f} thousand")
print("="*65)

---
## Step 18 — New House Prediction Visualization

We place the new house prediction on all three feature plots simultaneously. A large star marker shows where the new house sits relative to the full training and test set on each feature axis.

This answers the question: does the predicted price look reasonable given the house's area, bedroom count, and age compared to all other houses in the dataset?

In [ ]:
# New house prediction — visualized across all 3 features
new_area     = new_house[0][0]
new_bedrooms = new_house[0][1]
new_age      = new_house[0][2]
new_price    = predicted_price[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    ("Area",     new_area,     "Area (sq ft)"),
    ("Bedrooms", new_bedrooms, "Bedrooms"),
    ("Age",      new_age,      "Age (years)"),
]

for i, (feat, new_val, xlabel) in enumerate(configs):
    # All training houses
    axes[i].scatter(X_train[feat], Y_train,
                    color="steelblue", edgecolors="white",
                    s=60, linewidth=0.7, label="Train", zorder=2, alpha=0.7)
    # Test houses — actual
    axes[i].scatter(X_test[feat], Y_test,
                    color="darkorange", edgecolors="white",
                    s=80, linewidth=0.7, marker="D",
                    label="Test (actual)", zorder=3, alpha=0.9)
    # Test houses — predicted
    axes[i].scatter(X_test[feat], y_pred,
                    color="tomato", edgecolors="white",
                    s=80, linewidth=0.7, marker="^",
                    label="Test (predicted)", zorder=3, alpha=0.9)
    # New house star
    axes[i].scatter(new_val, new_price,
                    color="red", marker="*", s=400, zorder=5,
                    label=f"New house: {new_price:.0f}K")
    # Drop lines
    axes[i].vlines(new_val, 0, new_price,
                   colors="gray", linestyles="dotted", linewidth=1.2)
    axes[i].hlines(new_price, df[feat].min(), new_val,
                   colors="gray", linestyles="dotted", linewidth=1.2)

    axes[i].set_xlabel(xlabel, fontsize=11)
    axes[i].set_ylabel("Price (thousands)", fontsize=11)
    axes[i].set_title(f"{feat} — New House Highlighted", fontsize=11)
    axes[i].legend(fontsize=8)
    axes[i].grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    f"New House — Area: {new_area} sqft | Bedrooms: {new_bedrooms} "
    f"| Age: {new_age}yr | Predicted: {new_price:.2f}K",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()
print("="*65)

---
## Pipeline Summary

| Step | Action | Output |
|------|--------|--------|
| 1 | Import libraries | All tools ready |
| 2 | Create dataset | 25 houses, 3 features, 1 target |
| 3 | EDA — 3 scatter plots | Feature-target relationships visualized |
| 4 | Descriptive statistics and missing value check | Data quality confirmed |
| 5 | IQR outlier detection and clipping | Price column cleaned |
| 5b | Boxplot and histogram | Clipping confirmed visually |
| 6 | Correlation heatmap | Feature-target and inter-feature correlations |
| 7 | Feature and target selection | X shape (25,3), Y shape (25,) |
| 8 | Train-test split 80/20 | 20 training houses, 5 test houses |
| 9 | StandardScaler | All 3 features normalized, means and stds printed |
| 10 | LinearRegression trained | 3 weights + intercept learned, equation printed |
| 11 | Predictions + comparison table | y_pred with Actual / Predicted / Error per house |
| 12 | Evaluation — MSE, RMSE, MAE, R2 | Model performance quantified |
| 13 | Feature importance bar chart | Which feature contributes most to predictions |
| 14 | Post-training 3-panel scatter | Actual vs predicted across all 3 features |
| 15 | Actual vs Predicted plot | Diagonal cluster with R2 in title |
| 16 | Residual analysis | Residuals vs predicted + histogram |
| 17 | New house prediction | Price predicted for 2600 sqft, 4-bed, 5yr house |
| 18 | New house visualization | Star-marked across all 3 feature plots with drop lines |

---

**Key difference from Notebook 01 (Simple Linear Regression):**

| Aspect | Notebook 01 | Notebook 02 |
|--------|------------|-------------|
| Features | 1 (Area) | 3 (Area, Bedrooms, Age) |
| X shape | (20, 1) | (25, 3) |
| Weights learned | 1 | 3 |
| Extra steps | — | Correlation heatmap, feature importance chart |
| Expected R2 | Lower | Higher — more features explain more variance |

---

**Next notebook:** `03_logistic_regression.ipynb` — Binary Classification